---
## Section 5 — EEGNet in PyTorch

### Why CNNs Work on EEG

Traditional machine learning (CSP + SVM) requires **hand-crafted** spatial filters and log-variance features.
Convolutional neural networks can **learn** these transformations directly from raw data:

| CNN Operation | EEG Analogy | Why It Matters |
|---|---|---|
| **Temporal Conv2d** (1 × kernel_length) | Learned bandpass filter | Instead of manually choosing 8–30 Hz, the network discovers the optimal frequency bands from data — it may learn to weight mu and beta differently per subject |
| **Depthwise Conv2d** (n_channels × 1, groups=F1) | Learned spatial filter (like CSP) | Each filter learns a weighted combination of electrodes. `groups=F1` means each temporal filter gets its **own** spatial filter — no cross-filter mixing, dramatically fewer parameters than a standard Conv2d |
| **Pointwise Conv2d** (1 × 1) | Feature mixing across filter maps | After depthwise conv, each feature map is independent. The 1×1 conv mixes them — analogous to combining CSP components into a unified feature vector |

### Why Depthwise Convolution Instead of Regular Convolution

A standard Conv2d with F1 input filters and F2 output filters over (n_channels × 1) has
**F1 × F2 × n_channels** parameters. A depthwise conv with multiplier D has only
**F1 × D × n_channels** parameters — an **F2 / D** factor reduction. For EEG with 64 channels,
this prevents overfitting on the small datasets typical of BCI experiments (often < 200 trials).

### Why ELU Over ReLU

ReLU zeros out all negative activations, creating "dead neurons" that stop learning.
**ELU** (Exponential Linear Unit) allows small **negative outputs** (approaching −α for
very negative inputs), which:
- Keeps the mean activation closer to zero → faster convergence
- Prevents dead neurons → better gradient flow in the low-SNR EEG domain

### Why BatchNorm

EEG signals have large inter-trial and inter-subject amplitude variability (10–100 µV range).
**BatchNorm** normalizes each layer's input to zero mean and unit variance within each mini-batch,
which:
- Stabilizes gradient magnitudes → allows higher learning rates
- Acts as mild regularization → reduces overfitting on small BCI datasets

### Why EEGNet Specifically

EEGNet (Lawhern et al., 2018) was **designed for EEG** with these constraints in mind:
- **~2,500 parameters** for a 2-class, 64-channel model — orders of magnitude fewer than VGG/ResNet
- Embeds domain knowledge (temporal → spatial → separable conv) into the architecture
- Achieves competitive accuracy on datasets with as few as ~100 trials per class
- Architecture is interpretable: temporal filters ≈ bandpass filters, spatial filters ≈ CSP patterns

In [ ]:
# ============================================================
# SECTION 5 — Adaptive preprocessing for EEGNet
# ============================================================
# WHY: EEGNet learns filters end-to-end, so we keep preprocessing minimal
#      but rigorous. The key innovation here is *dynamic* ICA component
#      selection and multi-channel EOG correlation for robust artifact removal.

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split


def preprocess_for_eegnet(raw, sfreq_target=160, l_freq=1.0, h_freq=40.0,
                          epoch_tmin=0.5, epoch_tmax=4.5,
                          reject_uv=200e-6, event_id=None,
                          eog_threshold=0.4):
    """
    WHY:  An improved preprocessing pipeline that differs from Section 2's in three ways:
          1) Wider bandpass (1–40 Hz) — lets EEGNet learn its own optimal sub-bands
             instead of forcing 8–30 Hz; the network can discover gamma or theta
             components that CSP+SVM would miss.
          2) Dynamic ICA components — the number of components is set to
             min(n_channels - 1, 0.99 * n_samples / n_timepoints) following the
             rule-of-thumb that ICA needs ≥100 time-points per component for a
             stable unmixing matrix.  This avoids rank-deficient decompositions
             on short recordings and over-decomposition on small datasets.
          3) Multi-channel frontal EOG detection — instead of relying on a single
             EOG channel (which may not exist), we correlate every ICA component
             with *three* frontal channels (Fp1, Fz, Fp2) and flag any component
             whose maximum absolute correlation exceeds `eog_threshold`.  This
             catches both vertical blinks (Fp1/Fp2) and horizontal saccades (Fz
             asymmetry) more reliably than a single-channel approach.
    WHAT: Resample → widen bandpass → avg ref → adaptive ICA with multi-channel
          frontal EOG detection → epoch → baseline correction.
    INPUT:
        raw            : mne.io.Raw — raw EEG (preloaded)
        sfreq_target   : float — target sampling rate in Hz
        l_freq         : float — lower bandpass edge (1 Hz: keeps slow ERD onset)
        h_freq         : float — upper bandpass edge (40 Hz: keeps low-gamma)
        epoch_tmin     : float — epoch start relative to event (seconds)
        epoch_tmax     : float — epoch end relative to event (seconds)
        reject_uv      : float — peak-to-peak rejection threshold (volts)
        event_id       : dict or None — event label→code mapping
        eog_threshold  : float — correlation threshold for EOG detection (0–1)
    OUTPUT:
        epochs : mne.Epochs — cleaned, epoched data ready for EEGNet
    """
    raw = raw.copy()  # Preserve the original raw object

    if event_id is None:
        event_id = {'T1': 1, 'T2': 2}  # Left/right fist MI

    # ------------------------------------------------------------------
    # Step 1: Resample — reduce computation, still well above Nyquist for 40 Hz
    # ------------------------------------------------------------------
    current_sfreq = raw.info['sfreq']
    if abs(current_sfreq - sfreq_target) > 1.0:
        print(f"  Resampling: {current_sfreq:.0f} Hz → {sfreq_target} Hz")
        raw.resample(sfreq_target)

    # ------------------------------------------------------------------
    # Step 2: Wider bandpass 1–40 Hz (IIR Butterworth, order 5)
    # ------------------------------------------------------------------
    # WHY: Unlike the 8–30 Hz filter in Section 2, a 1–40 Hz pass-band
    #      preserves slow cortical potentials (1–4 Hz delta/theta that carry
    #      movement-preparation signals) and low-gamma activity (30–40 Hz).
    #      EEGNet's temporal convolution layer will learn to suppress
    #      irrelevant frequencies — acting as a data-driven bandpass.
    print(f"  Bandpass filtering: {l_freq}–{h_freq} Hz (IIR Butterworth)")
    raw.filter(l_freq=l_freq, h_freq=h_freq, method='iir',
               iir_params=dict(order=5, ftype='butter'))

    # ------------------------------------------------------------------
    # Step 3: Average reference — remove shared reference contamination
    # ------------------------------------------------------------------
    print("  Setting average reference")
    raw.set_eeg_reference('average', projection=True)
    raw.apply_proj()

    # ------------------------------------------------------------------
    # Step 4: Adaptive ICA with multi-channel frontal EOG detection
    # ------------------------------------------------------------------
    n_channels = len(raw.ch_names)
    n_samples = raw.n_times

    # Dynamic component count: enough for a well-conditioned unmixing matrix
    # Rule of thumb: need ≥ 20× more time-points than squared components
    # for a stable ICA. We use min(channels-1, int(sqrt(n_samples/20)))
    # but cap at channels-1 (the maximum possible rank).
    max_by_rank = n_channels - 1  # Maximum rank after average referencing
    max_by_data = int(0.99 * n_samples / max(sfreq_target, 1))  # ≈ samples / sfreq heuristic
    n_ica_components = min(max_by_rank, max(15, max_by_data))    # Floor at 15 components
    n_ica_components = min(n_ica_components, max_by_rank)         # Never exceed rank
    print(f"  ICA: using {n_ica_components} components "
          f"(rank={max_by_rank}, data-driven cap={max_by_data})")

    ica = mne.preprocessing.ICA(
        n_components=n_ica_components,
        method='fastica',
        random_state=42,
        max_iter=500
    )
    ica.fit(raw)

    # --- Multi-channel frontal EOG detection ---
    # WHY: Eye artifacts (blinks, saccades) are strongest at frontal electrodes.
    #      By correlating each ICA component with Fp1, Fz, AND Fp2 simultaneously,
    #      we catch both vertical blinks (symmetric Fp1/Fp2 deflection) and
    #      horizontal saccades (asymmetric Fp1 vs Fp2 pattern). A component is
    #      flagged as artifactual if its correlation with ANY of the three
    #      frontal channels exceeds the threshold.
    frontal_targets = ['Fp1', 'Fz', 'Fp2']  # Three frontal channels for EOG detection
    available_frontals = [ch for ch in frontal_targets if ch in raw.ch_names]

    eog_indices = set()  # Use a set to avoid duplicate indices

    if available_frontals:
        print(f"  Frontal channels found for EOG detection: {available_frontals}")
        # Get ICA source time-courses: shape (n_components, n_samples)
        sources = ica.get_sources(raw).get_data()

        for ch_name in available_frontals:
            # Extract the frontal channel's time-course
            ch_idx = raw.ch_names.index(ch_name)
            frontal_data = raw.get_data(picks=[ch_idx]).flatten()  # (n_samples,)

            # Correlate every ICA component with this frontal channel
            for comp_idx in range(sources.shape[0]):
                # Pearson correlation between component and frontal channel
                corr = np.abs(np.corrcoef(sources[comp_idx], frontal_data)[0, 1])
                if corr > eog_threshold:
                    eog_indices.add(comp_idx)
                    print(f"    Component {comp_idx} correlates with {ch_name}: "
                          f"r={corr:.3f} > {eog_threshold} → flagged as EOG")
    else:
        # Fallback: try MNE's built-in EOG detection
        print("  ⚠️ No frontal channels found — falling back to auto EOG detection")
        try:
            eog_idx, _ = ica.find_bads_eog(raw)
            eog_indices = set(eog_idx)
        except Exception:
            eog_indices = {0}  # Last resort: exclude component 0
            print("  ⚠️ Auto-detection failed — excluding component 0 as fallback")

    eog_indices = sorted(eog_indices)

    if len(eog_indices) == 0:
        print("  No EOG components detected above threshold — data appears clean")
    else:
        ica.exclude = eog_indices
        ica.apply(raw)
        print(f"  Removed {len(eog_indices)} EOG component(s): {eog_indices}")

    # ------------------------------------------------------------------
    # Step 5: Epoch extraction
    # ------------------------------------------------------------------
    print(f"  Epoching: [{epoch_tmin}, {epoch_tmax}]s")

    try:
        events_arr, _ = mne.events_from_annotations(raw, event_id=event_id)
    except ValueError:
        events_arr = mne.find_events(raw, shortest_event=1)

    epoch_event_id = {k: v for k, v in event_id.items() if v in events_arr[:, 2]}
    if len(epoch_event_id) == 0:
        raise ValueError("No matching events found in the data!")

    # Include a baseline window before epoch_tmin for baseline correction
    epochs = mne.Epochs(
        raw, events_arr, event_id=epoch_event_id,
        tmin=epoch_tmin - 0.5,  # 0.5s pre-epoch for baseline
        tmax=epoch_tmax,
        baseline=(epoch_tmin - 0.5, epoch_tmin),  # Baseline = 0.5s window before task onset
        reject=dict(eeg=reject_uv),
        preload=True,
        on_missing='warn'
    )

    # Crop to the analysis window after baseline correction is applied
    epochs.crop(tmin=epoch_tmin, tmax=epoch_tmax)

    n_dropped = len(events_arr) - len(epochs)
    print(f"  Epochs: {len(epochs)} kept, {n_dropped} rejected "
          f"(>{reject_uv*1e6:.0f} µV threshold)")

    return epochs


# --- Apply the new preprocessing to subject S001 ---
print("=" * 60)
print("Adaptive preprocessing for EEGNet — Subject S001")
print("=" * 60)

epochs_eegnet = preprocess_for_eegnet(raw, sfreq_target=160)

# Extract arrays in EEGNet format
X = epochs_eegnet.get_data()              # (n_epochs, n_channels, n_timepoints)
y = epochs_eegnet.events[:, 2] - 1        # Remap T1→0 (left), T2→1 (right)

print(f"\nX shape: {X.shape}  (n_epochs, n_channels, n_timepoints)")
print(f"y shape: {y.shape}")
print(f"Class 0 (left fist MI) : {np.sum(y == 0)}")
print(f"Class 1 (right fist MI): {np.sum(y == 1)}")

### What the Adaptive Preprocessing Does and Why

#### Pipeline Overview

| Step | What | Why it differs from Section 2 |
|------|------|-------------------------------|
| **Resampling** (160 Hz) | Down-sample to reduce computation | Same as Section 2 — Nyquist is satisfied for 40 Hz |
| **Bandpass 1–40 Hz** | Keep a wider frequency range than the 8–30 Hz used for CSP | EEGNet's temporal convolution layer acts as a *learned* bandpass filter — by giving it access to delta/theta (1–8 Hz) and low-gamma (30–40 Hz), we let the network discover informative frequency components that a hand-designed 8–30 Hz filter would discard |
| **Average reference** | Subtract the mean of all channels at each time point | Removes the shared reference electrode contamination, same rationale as Section 2 |
| **Dynamic ICA** | Compute the number of ICA components adaptively from the data dimensions | A fixed component count (e.g., 63) can cause rank-deficient decompositions on short recordings or over-decompose small datasets. The formula `min(rank, data-driven cap)` ensures a well-conditioned unmixing matrix every time |
| **Multi-channel frontal EOG** | Correlate each ICA component with **Fp1, Fz, and Fp2** | A single-channel EOG detector misses horizontal saccades (lateral Fp1/Fp2 asymmetry). By checking three frontal channels and using a correlation threshold, we catch both vertical blinks *and* horizontal eye movements without needing a dedicated EOG electrode |
| **Epoch extraction** | Segment continuous data into fixed-length windows around each event | Same purpose as Section 2, but with a wider pre-epoch baseline window for more stable baseline correction |

#### Why Dynamic ICA Component Selection?

ICA factorizes the data matrix $\mathbf{X} \in \mathbb{R}^{C \times T}$ (channels × time-points)
into an unmixing matrix $\mathbf{W}$ and independent sources $\mathbf{S} = \mathbf{W} \mathbf{X}$.
For a stable decomposition, we need far more time-points than components — a common rule of thumb is:

$$n_{\text{components}} \leq \frac{T}{k \cdot f_s}$$

where $T$ is the number of samples, $f_s$ is the sampling rate, and $k$ is a safety factor.
Our implementation uses $k \approx 1$ with a floor of 15 components, and caps at channel
rank − 1 (because average referencing reduces rank by one). This adaptive formula means:

- **Short recordings** → fewer components → avoids overfitting the ICA
- **Long recordings** → more components → captures finer source separation
- **Different channel counts** (64 vs 22) → automatically adjusts

#### Why Correlate with Fp1, Fz, and Fp2?

Eye artifacts have a characteristic spatial signature:

- **Blinks** produce large, symmetric deflections at **Fp1** and **Fp2** (directly above the eyes),
  with smaller amplitude at **Fz** (midline).
- **Horizontal saccades** produce an asymmetric pattern: positive at **Fp1** and negative at **Fp2**
  (or vice versa), with near-zero signal at **Fz**.

By correlating each ICA component with all three channels and flagging any component that exceeds
the threshold on *any* of them, we reliably detect both artifact types. The threshold of 0.4
(Pearson $|r|$) is conservative enough to avoid removing genuine brain components while catching
the high-amplitude ocular artifacts.

#### What Are Epochs and Why Are They Useful?

An **epoch** is a fixed-length time window extracted from the continuous EEG recording, aligned
to a specific event (e.g., the cue telling the subject to imagine moving their left or right hand).

**Why epoch?**
1. **Labeling**: Each epoch inherits the event label (left MI or right MI), creating the
   (input, label) pairs that supervised learning requires.
2. **Alignment**: By time-locking epochs to the event onset, the task-related brain activity
   (ERD/ERS) appears at consistent latencies across trials — making patterns learnable.
3. **Artifact rejection**: Epochs with extreme amplitude (> 200 µV) are automatically discarded,
   removing trials corrupted by muscle tension, electrode pops, or residual eye artifacts that
   ICA did not fully remove.
4. **Tensor format**: After epoching, the data is a 3D array `(n_epochs, n_channels, n_timepoints)`
   — exactly the format EEGNet expects (with an added batch dimension during training).

In this pipeline, each epoch spans **[0.5, 4.5] s** after the motor imagery cue, with a 0.5 s
pre-epoch window used solely for baseline correction (subtracting the mean pre-cue voltage so
that each epoch starts from a common zero level).

In [ ]:
# ============================================================
# SECTION 5 — EEGNet model definition (Lawhern et al. 2018)
# ============================================================

# Ensure reproducibility across all random number generators
np.random.seed(42)          # NumPy seed for data splitting
torch.manual_seed(42)       # PyTorch CPU seed for weight init
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)  # PyTorch GPU seed

from torch.utils.data import Dataset, DataLoader  # Dataset abstraction and batching


class EEGNet(nn.Module):
    """
    WHY:  EEGNet is a compact CNN designed for EEG-based BCIs. Its architecture
          mirrors classical EEG processing: temporal filtering → spatial filtering
          → feature extraction, but learns all stages end-to-end from data.
    WHAT: Two-block depthwise-separable CNN followed by a linear classification head.
          Block 1 learns temporal then spatial filters; Block 2 learns separable
          spatiotemporal features. Total ~2,500 params for 64-channel, 2-class.
    INPUT:
        x : torch.FloatTensor, shape (batch, 1, n_channels, n_timepoints)
    OUTPUT:
        logits : torch.FloatTensor, shape (batch, n_classes) — raw scores (no softmax)
    """

    def __init__(self, n_classes=2, n_channels=64, n_timepoints=640,
                 F1=8, D=2, F2=16, dropout_rate=0.5, kernel_length=64):
        super(EEGNet, self).__init__()

        # Store architecture hyperparameters for reference
        self.n_classes = n_classes
        self.n_channels = n_channels
        self.n_timepoints = n_timepoints
        self.F1 = F1              # Number of temporal filters in Block 1
        self.D = D                # Depth multiplier for depthwise conv
        self.F2 = F2              # Number of pointwise filters in Block 2 (should equal F1*D)
        self.dropout_rate = dropout_rate
        self.kernel_length = kernel_length  # Temporal filter length (samples); 64 @ 160 Hz ≈ 400 ms

        # ==============================================================
        # BLOCK 1: Temporal convolution → Depthwise spatial convolution
        # ==============================================================

        # Step 1a: Temporal convolution — learns F1 bandpass-like filters
        # Kernel (1, kernel_length) spans time only, applied identically to every channel.
        # padding='same' preserves the temporal dimension.
        self.conv1_temporal = nn.Conv2d(
            in_channels=1,            # Input has 1 "image" channel (the EEG tensor)
            out_channels=F1,          # Learn F1 temporal filters
            kernel_size=(1, kernel_length),  # 1 across channels, kernel_length across time
            padding='same',           # Zero-pad to keep temporal dim unchanged
            bias=False                # No bias — BatchNorm absorbs it
        )
        # BatchNorm after temporal conv stabilizes the learned filter outputs
        self.bn1 = nn.BatchNorm2d(F1)

        # Step 1b: Depthwise convolution — learns spatial filters per temporal filter
        # Kernel (n_channels, 1) spans all channels at a single time point.
        # groups=F1 means each temporal filter gets D independent spatial filters,
        # with NO mixing between different temporal filters (depthwise constraint).
        # This is the key efficiency: F1*D*n_channels params vs F1*F2*n_channels for standard conv.
        self.conv1_depthwise = nn.Conv2d(
            in_channels=F1,                 # One spatial filter per temporal filter
            out_channels=F1 * D,            # D spatial filters per temporal filter
            kernel_size=(n_channels, 1),     # Full spatial extent, 1 time sample
            groups=F1,                       # Depthwise: each input channel group is filtered independently
            bias=False                       # No bias — BatchNorm absorbs it
        )
        # BatchNorm normalizes the spatial filter outputs
        self.bn2 = nn.BatchNorm2d(F1 * D)
        # ELU allows small negative outputs → prevents dead neurons in low-SNR EEG
        self.elu1 = nn.ELU()
        # AvgPool reduces temporal dimension by 4× → downsamples to ~40 Hz effective rate
        self.pool1 = nn.AvgPool2d(kernel_size=(1, 4))
        # Dropout prevents co-adaptation of spatial filter weights
        self.drop1 = nn.Dropout(p=dropout_rate)

        # ==============================================================
        # BLOCK 2: Separable convolution (depthwise + pointwise)
        # ==============================================================

        # Step 2a: Depthwise temporal convolution — further temporal feature extraction
        # Each of the F1*D feature maps gets its own temporal filter of length 16.
        # groups=F1*D ensures no cross-channel mixing at this stage.
        self.conv2_depthwise = nn.Conv2d(
            in_channels=F1 * D,             # Input from Block 1
            out_channels=F1 * D,            # Same number of feature maps
            kernel_size=(1, 16),            # Temporal kernel length 16 (≈100 ms @ 160 Hz)
            padding='same',                 # Preserve temporal dimension
            groups=F1 * D,                  # Depthwise: each feature map filtered independently
            bias=False                      # No bias — BatchNorm absorbs it
        )

        # Step 2b: Pointwise convolution (1×1) — mixes features across filter maps
        # This is where the network learns which combinations of the depthwise features
        # are useful for classification — analogous to combining CSP components.
        self.conv2_pointwise = nn.Conv2d(
            in_channels=F1 * D,             # Input from depthwise step
            out_channels=F2,                # Output F2 mixed feature maps
            kernel_size=(1, 1),             # 1×1 conv: mix channels, no spatial/temporal extent
            bias=False                      # No bias — BatchNorm absorbs it
        )
        # BatchNorm after pointwise conv
        self.bn3 = nn.BatchNorm2d(F2)
        # ELU activation
        self.elu2 = nn.ELU()
        # AvgPool reduces temporal dimension by 8× → aggressive downsampling
        self.pool2 = nn.AvgPool2d(kernel_size=(1, 8))
        # Dropout for regularization
        self.drop2 = nn.Dropout(p=dropout_rate)

        # ==============================================================
        # Compute flattened feature size via dummy forward pass
        # ==============================================================
        # WHY: The output size depends on input dimensions and pooling layers.
        #      A dummy pass avoids error-prone manual calculation.
        self._flat_size = self._get_flat_size(n_channels, n_timepoints)

        # ==============================================================
        # CLASSIFICATION HEAD
        # ==============================================================
        # Linear layer maps flattened features to class logits.
        # No softmax here — CrossEntropyLoss includes it internally.
        self.classifier = nn.Linear(self._flat_size, n_classes)

        # Print architecture summary with per-layer output shapes
        self._print_architecture(n_channels, n_timepoints)

    def _get_flat_size(self, n_channels, n_timepoints):
        """
        WHY:  Automatically compute the flattened feature dimension after all conv/pool layers.
        WHAT: Run a dummy tensor through the feature extractor and measure output size.
        INPUT:
            n_channels   : int — number of EEG channels
            n_timepoints : int — number of time samples per epoch
        OUTPUT:
            flat_size : int — total number of features after flattening
        """
        # Create a dummy input: batch=1, 1 image channel, n_channels × n_timepoints
        dummy = torch.zeros(1, 1, n_channels, n_timepoints)
        # Pass through all layers except classifier
        x = self._forward_features(dummy)
        # Flatten and return the total feature count
        return x.view(1, -1).size(1)

    def _forward_features(self, x):
        """
        WHY:  Separating feature extraction from classification enables reuse
              (e.g., for flat_size computation, feature visualization).
        WHAT: Pass input through Blocks 1 and 2 (conv → bn → elu → pool → dropout).
        INPUT:
            x : torch.FloatTensor, shape (batch, 1, n_channels, n_timepoints)
        OUTPUT:
            x : torch.FloatTensor, shape (batch, F2, 1, reduced_timepoints)
        """
        # --- Block 1 ---
        x = self.conv1_temporal(x)    # (batch, F1, n_channels, n_timepoints) — temporal filtering
        x = self.bn1(x)              # Normalize temporal filter outputs
        x = self.conv1_depthwise(x)  # (batch, F1*D, 1, n_timepoints) — spatial filtering collapses channel dim
        x = self.bn2(x)              # Normalize spatial filter outputs
        x = self.elu1(x)             # Non-linear activation (ELU preserves negative gradients)
        x = self.pool1(x)            # (batch, F1*D, 1, n_timepoints//4) — temporal downsampling
        x = self.drop1(x)            # Regularization: randomly zero 50% of activations

        # --- Block 2 ---
        x = self.conv2_depthwise(x)  # (batch, F1*D, 1, n_timepoints//4) — temporal refinement
        x = self.conv2_pointwise(x)  # (batch, F2, 1, n_timepoints//4) — cross-filter mixing
        x = self.bn3(x)              # Normalize mixed features
        x = self.elu2(x)             # Non-linear activation
        x = self.pool2(x)            # (batch, F2, 1, n_timepoints//32) — aggressive downsampling
        x = self.drop2(x)            # Regularization

        return x

    def forward(self, x):
        """
        WHY:  Full forward pass for training and inference.
        WHAT: Feature extraction → flatten → linear classifier → raw logits.
        INPUT:
            x : torch.FloatTensor, shape (batch, 1, n_channels, n_timepoints)
        OUTPUT:
            logits : torch.FloatTensor, shape (batch, n_classes)
        """
        x = self._forward_features(x)     # Extract spatiotemporal features
        x = x.view(x.size(0), -1)         # Flatten: (batch, F2 * 1 * reduced_time)
        x = self.classifier(x)            # Map to class logits (no softmax — CE loss handles it)
        return x

    def get_n_params(self):
        """
        WHY:  Track model complexity — EEGNet's key advantage is having very few parameters.
        WHAT: Sum all trainable parameter counts across every layer.
        INPUT:  None
        OUTPUT: int — total number of trainable parameters
        """
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def _print_architecture(self, n_channels, n_timepoints):
        """
        WHY:  Verify that each layer produces the expected output shape.
        WHAT: Run a dummy forward pass and print layer-by-layer shapes.
        INPUT:
            n_channels   : int — number of EEG channels
            n_timepoints : int — number of time samples
        OUTPUT: None (prints to stdout)
        """
        print("\n" + "=" * 65)
        print("EEGNet Architecture Summary")
        print("=" * 65)

        dummy = torch.zeros(1, 1, n_channels, n_timepoints)
        shapes = [('Input', dummy.shape)]

        # Block 1
        dummy = self.conv1_temporal(dummy)
        shapes.append(('Conv2d_temporal', dummy.shape))
        dummy = self.bn1(dummy)
        shapes.append(('BatchNorm2d_1', dummy.shape))
        dummy = self.conv1_depthwise(dummy)
        shapes.append(('Conv2d_depthwise_spatial', dummy.shape))
        dummy = self.bn2(dummy)
        shapes.append(('BatchNorm2d_2', dummy.shape))
        dummy = self.elu1(dummy)
        shapes.append(('ELU_1', dummy.shape))
        dummy = self.pool1(dummy)
        shapes.append(('AvgPool2d_1 (1,4)', dummy.shape))
        dummy = self.drop1(dummy)
        shapes.append(('Dropout_1', dummy.shape))

        # Block 2
        dummy = self.conv2_depthwise(dummy)
        shapes.append(('Conv2d_depthwise_temp', dummy.shape))
        dummy = self.conv2_pointwise(dummy)
        shapes.append(('Conv2d_pointwise (1x1)', dummy.shape))
        dummy = self.bn3(dummy)
        shapes.append(('BatchNorm2d_3', dummy.shape))
        dummy = self.elu2(dummy)
        shapes.append(('ELU_2', dummy.shape))
        dummy = self.pool2(dummy)
        shapes.append(('AvgPool2d_2 (1,8)', dummy.shape))
        dummy = self.drop2(dummy)
        shapes.append(('Dropout_2', dummy.shape))

        # Head
        flat = dummy.view(1, -1)
        shapes.append(('Flatten', flat.shape))
        out = self.classifier(flat)
        shapes.append(('Linear (classifier)', out.shape))

        for name, shape in shapes:
            print(f"  {name:<30s} → {str(list(shape)):>25s}")

        print(f"\n  Total trainable parameters: {self.get_n_params():,}")
        print("=" * 65)


# --- Instantiate EEGNet on subject S001 data to verify architecture ---
# X was defined in Section 2: shape (n_epochs, 64, n_timepoints)
n_channels_s001 = X.shape[1]       # 64 channels for Schalk dataset
n_timepoints_s001 = X.shape[2]     # Number of time samples per epoch

model_demo = EEGNet(
    n_classes=2,
    n_channels=n_channels_s001,
    n_timepoints=n_timepoints_s001,
    F1=8, D=2, F2=16,
    dropout_rate=0.5,
    kernel_length=64               # 64 samples @ 160 Hz ≈ 400 ms temporal filter
).to(device)

print(f"\nModel on device: {device}")
print(f"Input shape expected: (batch, 1, {n_channels_s001}, {n_timepoints_s001})")

In [ ]:
# ============================================================
# SECTION 5 — EEGDataset + Data splitting + DataLoaders
# ============================================================

class EEGDataset(Dataset):
    """
    WHY:  PyTorch requires a Dataset object for efficient batched data loading.
          Per-epoch per-channel normalization removes amplitude variability
          across trials and channels, ensuring the network sees standardized
          inputs regardless of electrode impedance or cortical source depth.
    WHAT: Wraps raw EEG numpy arrays into normalized PyTorch tensors with the
          extra channel dimension EEGNet expects.
    INPUT:
        X : np.ndarray, shape (n_epochs, n_channels, n_timepoints) — raw EEG
        y : np.ndarray, shape (n_epochs,) — integer class labels (0 or 1)
    OUTPUT (per __getitem__):
        x_tensor : torch.FloatTensor, shape (1, n_channels, n_timepoints)
        y_tensor : torch.LongTensor, scalar — class label
    """

    def __init__(self, X, y):
        # Normalize each epoch independently: zero mean, unit std per channel
        # WHY: Removes amplitude scale differences across trials/channels.
        #      eps=1e-8 prevents division by zero for flat (dead) channels.
        eps = 1e-8  # Small constant for numerical stability
        mean = X.mean(axis=2, keepdims=True)   # Mean over time: (n_epochs, n_channels, 1)
        std = X.std(axis=2, keepdims=True)      # Std over time: (n_epochs, n_channels, 1)
        self.X = (X - mean) / (std + eps)       # Z-score normalize per epoch per channel

        self.y = y.astype(np.int64)  # LongTensor required by CrossEntropyLoss

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        # Add the "image channel" dimension: (n_channels, n_timepoints) → (1, n_channels, n_timepoints)
        # WHY: Conv2d expects 4D input (batch, channels, height, width). The "1" is the image channel.
        x_tensor = torch.FloatTensor(self.X[idx][np.newaxis, :, :])  # (1, n_ch, n_time)
        y_tensor = torch.LongTensor([self.y[idx]])[0]               # Scalar LongTensor
        return x_tensor, y_tensor


# --- Stratified 70 / 15 / 15 split ---
# WHY: Stratified splitting preserves class balance in each partition.
#      70/15/15 gives enough training data while keeping meaningful val and test sets.
from sklearn.model_selection import train_test_split

# First split: 70% train vs 30% temp (will become 15% val + 15% test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,           # 30% held out
    stratify=y,               # Preserve class ratio in each split
    random_state=42           # Reproducibility
)

# Second split: 50% of temp → 15% val + 15% test (relative to original)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,           # Half of 30% = 15% each
    stratify=y_temp,          # Preserve class ratio
    random_state=42
)

# Create Dataset objects (normalization happens inside EEGDataset.__init__)
train_dataset = EEGDataset(X_train, y_train)
val_dataset = EEGDataset(X_val, y_val)
test_dataset = EEGDataset(X_test, y_test)

# Create DataLoaders for batched iteration
# WHY: batch_size=32 balances gradient noise (too small) vs memory (too large).
#      shuffle=True for train randomizes sample order each epoch → reduces overfitting.
#      Val/test loaders don't shuffle to keep results deterministic.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, drop_last=False)

# --- Print split summary ---
print("Data split summary (stratified 70 / 15 / 15):")
print(f"  Train : {len(train_dataset):>4d} epochs  |  Class 0: {np.sum(y_train==0):>3d}  Class 1: {np.sum(y_train==1):>3d}  |  ratio: {np.mean(y_train==1):.2f}")
print(f"  Val   : {len(val_dataset):>4d} epochs  |  Class 0: {np.sum(y_val==0):>3d}  Class 1: {np.sum(y_val==1):>3d}  |  ratio: {np.mean(y_val==1):.2f}")
print(f"  Test  : {len(test_dataset):>4d} epochs  |  Class 0: {np.sum(y_test==0):>3d}  Class 1: {np.sum(y_test==1):>3d}  |  ratio: {np.mean(y_test==1):.2f}")
print(f"  Total : {len(train_dataset)+len(val_dataset)+len(test_dataset)} epochs")

# Verify tensor shapes from the first batch
sample_x, sample_y = next(iter(train_loader))
print(f"\nSample batch — X: {sample_x.shape}, y: {sample_y.shape}")
print(f"  X dtype: {sample_x.dtype}, y dtype: {sample_y.dtype}")

In [ ]:
# ============================================================
# SECTION 5 — Training function with early stopping
# ============================================================

def train_eegnet(model, train_loader, val_loader, n_epochs=300,
                 lr=0.001, patience=20, device='cpu'):
    """
    WHY:  Train EEGNet with best practices for small EEG datasets:
          - Adam optimizer adapts learning rate per parameter → faster convergence
          - Weight decay (L2 reg) penalizes large weights → reduces overfitting
          - ReduceLROnPlateau halves LR when val_loss plateaus → finer optimization
          - Early stopping restores the best weights → prevents overfitting
    WHAT: Train loop with per-epoch val evaluation, LR scheduling, and early stopping.
    INPUT:
        model        : nn.Module — EEGNet instance (already on device)
        train_loader : DataLoader — training data batches
        val_loader   : DataLoader — validation data batches
        n_epochs     : int — maximum training epochs (300)
        lr           : float — initial learning rate (0.001)
        patience     : int — early stopping patience (20 epochs)
        device       : str — 'cpu' or 'cuda'
    OUTPUT:
        model      : nn.Module — trained model with best weights restored
        history    : dict — keys: 'train_loss', 'val_loss', 'train_acc', 'val_acc' (lists)
        best_epoch : int — epoch index (0-based) with lowest val_loss
    """
    import copy  # For deep-copying best model weights

    # CrossEntropyLoss combines LogSoftmax + NLLLoss — standard for classification
    criterion = nn.CrossEntropyLoss()

    # Adam optimizer with weight decay (L2 regularization)
    # WHY: weight_decay=1e-4 penalizes large weights, acting as L2 regularization
    #      to prevent overfitting on the small EEG training set.
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    # ReduceLROnPlateau: halve LR when val_loss hasn't improved for 10 epochs
    # WHY: As training converges, a smaller LR allows finer weight adjustments
    #      without oscillating around the minimum.
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min',      # Monitor val_loss (lower is better)
        factor=0.5,                 # Multiply LR by 0.5 on plateau
        patience=10,                # Wait 10 epochs before reducing
        verbose=False               # Suppress LR change messages
    )

    # History tracking
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    # Early stopping state
    best_val_loss = float('inf')  # Track the lowest val_loss seen
    best_epoch = 0                # Epoch with lowest val_loss
    best_weights = None           # Deep copy of model weights at best epoch
    epochs_no_improve = 0         # Counter for patience

    for epoch in range(n_epochs):
        # ---- Training phase ----
        model.train()  # Enable dropout and BatchNorm training mode
        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)  # Move input to GPU/CPU
            y_batch = y_batch.to(device)  # Move labels to GPU/CPU

            optimizer.zero_grad()          # Clear accumulated gradients from previous step
            logits = model(X_batch)        # Forward pass: (batch, n_classes)
            loss = criterion(logits, y_batch)  # Compute cross-entropy loss
            loss.backward()                # Backprop: compute gradients for all parameters
            optimizer.step()               # Update weights using Adam rule

            train_loss_sum += loss.item() * X_batch.size(0)  # Accumulate weighted loss
            preds = logits.argmax(dim=1)   # Predicted class = highest logit
            train_correct += (preds == y_batch).sum().item()  # Count correct predictions
            train_total += X_batch.size(0)

        train_loss = train_loss_sum / train_total  # Average loss over all training samples
        train_acc = train_correct / train_total     # Training accuracy

        # ---- Validation phase ----
        model.eval()  # Disable dropout, use running stats for BatchNorm
        val_loss_sum = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():  # No gradient computation → saves memory and time
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                logits = model(X_batch)            # Forward pass only
                loss = criterion(logits, y_batch)  # Validation loss

                val_loss_sum += loss.item() * X_batch.size(0)
                preds = logits.argmax(dim=1)
                val_correct += (preds == y_batch).sum().item()
                val_total += X_batch.size(0)

        val_loss = val_loss_sum / val_total  # Average validation loss
        val_acc = val_correct / val_total     # Validation accuracy

        # Record metrics for this epoch
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        # Step the learning rate scheduler based on validation loss
        scheduler.step(val_loss)

        # ---- Early stopping check ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_weights = copy.deepcopy(model.state_dict())  # Save best weights
            epochs_no_improve = 0  # Reset patience counter
        else:
            epochs_no_improve += 1  # No improvement this epoch

        # Print progress every 25 epochs or at early stop
        if (epoch + 1) % 25 == 0 or epochs_no_improve == patience:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"  Epoch {epoch+1:>3d}/{n_epochs} | "
                  f"Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | "
                  f"Val Loss: {val_loss:.4f} Acc: {val_acc:.3f} | "
                  f"LR: {current_lr:.6f}")

        # Stop if no improvement for `patience` consecutive epochs
        if epochs_no_improve >= patience:
            print(f"\n  Early stopping at epoch {epoch+1}. "
                  f"Best epoch: {best_epoch+1} (val_loss={best_val_loss:.4f})")
            break

    # Restore the best model weights (from the epoch with lowest val_loss)
    if best_weights is not None:
        model.load_state_dict(best_weights)
        print(f"  Restored best weights from epoch {best_epoch+1}")

    return model, history, best_epoch


# --- Train EEGNet on subject S001 ---
print("=" * 60)
print("Training EEGNet on subject S001")
print("=" * 60)

# Instantiate a fresh model for training
model_s001 = EEGNet(
    n_classes=2,
    n_channels=n_channels_s001,
    n_timepoints=n_timepoints_s001,
    F1=8, D=2, F2=16,
    dropout_rate=0.5,
    kernel_length=64
).to(device)

# Train with early stopping
model_s001, history_s001, best_epoch_s001 = train_eegnet(
    model_s001, train_loader, val_loader,
    n_epochs=300, lr=0.001, patience=20, device=device
)

In [ ]:
# ============================================================
# SECTION 5 — Plot training curves (loss + accuracy)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history_s001['train_loss']) + 1)  # 1-indexed epoch numbers

# --- Left panel: Loss curves ---
ax = axes[0]
ax.plot(epochs_range, history_s001['train_loss'], 'b-', linewidth=1.5, label='Train loss')
ax.plot(epochs_range, history_s001['val_loss'], 'r-', linewidth=1.5, label='Val loss')
# Mark the best epoch with a vertical dashed line
ax.axvline(best_epoch_s001 + 1, color='green', linestyle='--', linewidth=1.5,
           label=f'Best epoch ({best_epoch_s001 + 1})')  # +1 for 1-indexed display
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax.set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# --- Right panel: Accuracy curves ---
ax = axes[1]
ax.plot(epochs_range, history_s001['train_acc'], 'b-', linewidth=1.5, label='Train accuracy')
ax.plot(epochs_range, history_s001['val_acc'], 'r-', linewidth=1.5, label='Val accuracy')
# Mark the best epoch
ax.axvline(best_epoch_s001 + 1, color='green', linestyle='--', linewidth=1.5,
           label=f'Best epoch ({best_epoch_s001 + 1})')
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, label='Chance (50%)')  # Chance level
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Training & Validation Accuracy', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/section5_fig1.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/section5_fig1.png")

### What to look for in this plot:

- **Convergence**: Both train and val loss should decrease and stabilize. If train loss keeps
  dropping but val loss rises, the model is **overfitting** (memorizing training data).
- **Gap between curves**: A large train-val gap indicates overfitting. EEGNet's small parameter
  count and dropout should keep this gap modest.
- **Best epoch (green dashed line)**: Early stopping should trigger before the val loss diverges.
  The green line marks when the best weights were saved.
- **Accuracy above chance (50%)**: Both curves should rise above the gray chance line.
  Val accuracy at the best epoch is the model's estimated generalization performance.
- **Learning rate reductions**: Look for kinks in the loss curve where ReduceLROnPlateau
  halved the learning rate, allowing finer optimization.

In [ ]:
# ============================================================
# SECTION 5 — Test-set evaluation: accuracy, F1, confusion matrix
# ============================================================

def evaluate_model(model, data_loader, device='cpu'):
    """
    WHY:  Evaluate trained model on held-out data to estimate real-world performance.
    WHAT: Run inference on all batches, collect predictions and probabilities.
    INPUT:
        model       : nn.Module — trained EEGNet
        data_loader : DataLoader — test or validation data
        device      : str — 'cpu' or 'cuda'
    OUTPUT:
        y_true  : np.ndarray, shape (n_samples,) — ground truth labels
        y_pred  : np.ndarray, shape (n_samples,) — predicted labels
        y_proba : np.ndarray, shape (n_samples, n_classes) — softmax probabilities
    """
    model.eval()  # Disable dropout, use running BatchNorm stats
    all_true, all_pred, all_proba = [], [], []

    with torch.no_grad():  # No gradient tracking → faster inference
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)                         # Raw scores: (batch, n_classes)
            proba = torch.softmax(logits, dim=1)            # Convert to probabilities
            preds = logits.argmax(dim=1)                    # Predicted class

            all_true.append(y_batch.numpy())                # Keep on CPU
            all_pred.append(preds.cpu().numpy())            # Move preds to CPU
            all_proba.append(proba.cpu().numpy())           # Move probas to CPU

    return np.concatenate(all_true), np.concatenate(all_pred), np.concatenate(all_proba)


# Evaluate on test set
y_true_test, y_pred_test, y_proba_test = evaluate_model(model_s001, test_loader, device=device)

# Compute metrics
test_acc = accuracy_score(y_true_test, y_pred_test)
test_f1 = f1_score(y_true_test, y_pred_test, average='macro')  # Macro: unweighted mean across classes

print("=" * 50)
print("Test Set Evaluation — Subject S001")
print("=" * 50)
print(f"  Accuracy : {test_acc:.3f}")
print(f"  F1 (macro): {test_f1:.3f}")

# --- Confusion matrix (normalized) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Normalized confusion matrix: each row sums to 1.0
cm = confusion_matrix(y_true_test, y_pred_test, normalize='true')
sns.heatmap(
    cm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=['Left MI', 'Right MI'],
    yticklabels=['Left MI', 'Right MI'],
    ax=axes[0],
    cbar_kws={'label': 'Proportion'},
    annot_kws={'size': 16}
)
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('True', fontsize=12)
axes[0].set_title(f'EEGNet Confusion Matrix — S001 (Acc={test_acc:.2f})',
                  fontsize=13, fontweight='bold')

# --- ROC Curve with AUC ---
# Use probability of class 1 (right MI) for ROC
fpr, tpr, _ = roc_curve(y_true_test, y_proba_test[:, 1])
roc_auc = roc_auc_score(y_true_test, y_proba_test[:, 1])

axes[1].plot(fpr, tpr, 'b-', linewidth=2.5, label=f'EEGNet ROC (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'r--', linewidth=1.5, label='Chance (AUC = 0.5)')  # Diagonal = random
axes[1].fill_between(fpr, 0, tpr, alpha=0.15, color='blue')  # Shade area under curve
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title('ROC Curve — S001', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].set_aspect('equal')  # Square aspect ratio for proper ROC visualization
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/section5_fig2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/section5_fig2.png")

### What to look for in this plot:

- **Confusion matrix diagonals**: Values > 0.6 on the diagonal indicate the model learned
  meaningful left vs right MI discrimination. Off-diagonal = misclassification rate.
- **Class imbalance in errors**: If one class is consistently misclassified more, the model
  may be biased — check if class balance in training was preserved.
- **ROC curve**: Should bow toward upper-left. AUC > 0.6 is above chance; > 0.75 is good
  for single-subject EEG decoding.
- **EEGNet vs CSP+SVM**: Compare these metrics with Section 4's confusion matrix and ROC
  to see if the deep learning approach improves discrimination.

In [ ]:
# ============================================================
# SECTION 5 — ROC curve standalone (section5_fig3.png)
# ============================================================

fig, ax = plt.subplots(figsize=(7, 7))

# Plot ROC with confidence-band style shading
ax.plot(fpr, tpr, 'b-', linewidth=2.5, label=f'EEGNet (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'r--', linewidth=1.5, label='Chance (AUC = 0.5)')
ax.fill_between(fpr, 0, tpr, alpha=0.12, color='blue')  # Area under curve shading

# Mark the operating point closest to the top-left corner (optimal threshold)
# WHY: The point closest to (0,1) maximizes sensitivity while minimizing false positives
optimal_idx = np.argmax(tpr - fpr)  # Youden's J statistic
ax.plot(fpr[optimal_idx], tpr[optimal_idx], 'go', markersize=12,
        label=f'Optimal threshold (FPR={fpr[optimal_idx]:.2f}, TPR={tpr[optimal_idx]:.2f})')

ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('EEGNet ROC Curve — Subject S001', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/section5_fig3.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/section5_fig3.png")

### What to look for in this plot:

- **Curve shape**: A curve hugging the top-left corner indicates strong discrimination.
  A curve near the diagonal means the model is no better than random.
- **Optimal threshold (green dot)**: This is the decision threshold that maximizes
  Youden's J statistic (TPR − FPR). In a clinical BCI, you might choose a different
  threshold depending on whether false positives or false negatives are more costly.
- **AUC value**: 0.5 = random, 0.7 = acceptable, 0.8 = good, 0.9+ = excellent for BCI.

In [ ]:
# ============================================================
# SECTION 5 — Multi-subject EEGNet on Schalk (S001–S109)
# ============================================================

def train_eegnet_for_subject(X_subj, y_subj, n_channels, n_timepoints,
                              n_epochs=300, lr=0.001, patience=20, device='cpu'):
    """
    WHY:  Encapsulates the full EEGNet pipeline for a single subject: split, build,
          train, evaluate. Called in a loop for multi-subject analysis.
    WHAT: Stratified split → DataLoaders → fresh EEGNet → train → test accuracy.
    INPUT:
        X_subj       : np.ndarray, shape (n_epochs, n_channels, n_timepoints)
        y_subj       : np.ndarray, shape (n_epochs,) — labels 0 or 1
        n_channels   : int — number of EEG channels
        n_timepoints : int — time samples per epoch
        n_epochs     : int — max training epochs
        lr           : float — learning rate
        patience     : int — early stopping patience
        device       : str — 'cpu' or 'cuda'
    OUTPUT:
        test_acc : float — test set accuracy
    """
    # Stratified 70/15/15 split
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X_subj, y_subj, test_size=0.30, stratify=y_subj, random_state=42)
    X_v, X_te, y_v, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42)

    # Create DataLoaders
    tr_loader = DataLoader(EEGDataset(X_tr, y_tr), batch_size=32, shuffle=True)
    vl_loader = DataLoader(EEGDataset(X_v, y_v),   batch_size=32, shuffle=False)
    te_loader = DataLoader(EEGDataset(X_te, y_te), batch_size=32, shuffle=False)

    # Fresh EEGNet instance — each subject gets its own model
    # WHY: Subject-specific models account for individual cortical anatomy
    model = EEGNet(
        n_classes=2, n_channels=n_channels, n_timepoints=n_timepoints,
        F1=8, D=2, F2=16, dropout_rate=0.5, kernel_length=64
    ).to(device)

    # Train with early stopping (suppress per-epoch output for multi-subject loop)
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):  # Silence training output
        model, _, _ = train_eegnet(model, tr_loader, vl_loader,
                                   n_epochs=n_epochs, lr=lr,
                                   patience=patience, device=device)

    # Evaluate on test set
    y_true, y_pred, _ = evaluate_model(model, te_loader, device=device)
    test_acc = accuracy_score(y_true, y_pred)
    return test_acc


# ---- Run EEGNet on all 109 Schalk subjects ----
print("=" * 65)
print("Multi-subject EEGNet analysis: Schalk S001–S109")
print("=" * 65)

subject_accuracies_eegnet_schalk = {}  # {subject_id_str: accuracy}

for sid in range(1, 110):  # Subjects 1–109
    sub_label = f'S{sid:03d}'
    try:
        # Load and concatenate MI runs (4, 8, 12) for this subject
        raw_subj = load_schalk_subject(subject_id=sid, runs=[4, 8, 12])

        # Apply the same preprocessing pipeline from Section 2
        epochs_subj = preprocess_raw(raw_subj, sfreq_target=160)

        # Extract arrays
        X_subj = epochs_subj.get_data()             # (n_epochs, 64, n_timepoints)
        y_subj = epochs_subj.events[:, 2] - 1       # Remap T1→0, T2→1

        # Skip subjects with too few trials per class (need ≥6 for stratified split)
        class_counts = Counter(y_subj)
        if min(class_counts.values()) < 6:
            print(f"  {sub_label}: Skipped — too few trials {dict(class_counts)}")
            continue

        # Train EEGNet from scratch on this subject
        acc = train_eegnet_for_subject(
            X_subj, y_subj,
            n_channels=X_subj.shape[1],
            n_timepoints=X_subj.shape[2],
            n_epochs=300, lr=0.001, patience=20, device=device
        )
        subject_accuracies_eegnet_schalk[sub_label] = acc
        print(f"  {sub_label}: EEGNet accuracy = {acc:.3f}")

    except Exception as e:
        print(f"  {sub_label}: FAILED — {e}")

print(f"\nProcessed {len(subject_accuracies_eegnet_schalk)} / 109 subjects")
eegnet_accs = list(subject_accuracies_eegnet_schalk.values())
print(f"EEGNet mean accuracy: {np.mean(eegnet_accs):.3f} ± {np.std(eegnet_accs):.3f}")

In [ ]:
# ============================================================
# SECTION 5 — Grouped bar chart: CSP+SVM vs EEGNet (Schalk)
# ============================================================

from scipy.stats import ttest_rel  # Paired t-test for matched subject comparisons

# Find subjects with results from BOTH methods
# WHY: Paired t-test requires the same subjects in both groups
common_subjects = sorted(
    set(subject_accuracies_csp.keys()) & set(subject_accuracies_eegnet_schalk.keys())
)

csp_accs_common = [subject_accuracies_csp[s] for s in common_subjects]   # CSP+SVM accuracies
eeg_accs_common = [subject_accuracies_eegnet_schalk[s] for s in common_subjects] # EEGNet accuracies

# --- Grouped bar chart ---
fig, ax = plt.subplots(figsize=(max(14, len(common_subjects) * 0.6), 6))

x_pos = np.arange(len(common_subjects))  # Bar positions
bar_width = 0.35  # Width of each bar

# Plot CSP+SVM bars (left) and EEGNet bars (right) side by side
bars_csp = ax.bar(x_pos - bar_width/2, csp_accs_common, bar_width,
                  color='#3498db', edgecolor='black', linewidth=0.5, label='CSP+SVM')
bars_eeg = ax.bar(x_pos + bar_width/2, eeg_accs_common, bar_width,
                  color='#e74c3c', edgecolor='black', linewidth=0.5, label='EEGNet')

# Reference lines
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1.5, label='Chance (50%)')  # Random baseline
ax.axhline(np.mean(csp_accs_common), color='#3498db', linestyle=':',
           linewidth=1.5, label=f'CSP mean ({np.mean(csp_accs_common):.2f})')
ax.axhline(np.mean(eeg_accs_common), color='#e74c3c', linestyle=':',
           linewidth=1.5, label=f'EEGNet mean ({np.mean(eeg_accs_common):.2f})')

ax.set_xlabel('Subject', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('CSP+SVM vs EEGNet — Per-Subject Accuracy (Schalk Dataset)',
             fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(common_subjects, rotation=45, ha='right', fontsize=8)
ax.legend(fontsize=9, loc='upper right')
ax.set_ylim(0.3, 1.0)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/section5_fig4.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/section5_fig4.png")

# --- Paired t-test: CSP+SVM vs EEGNet ---
# WHY: A paired t-test accounts for inter-subject variability by comparing
#      each subject's two scores directly. This is more powerful than an
#      independent t-test when the same subjects are measured twice.
t_stat, p_value = ttest_rel(eeg_accs_common, csp_accs_common)
print(f"\nPaired t-test (EEGNet vs CSP+SVM) on {len(common_subjects)} subjects:")
print(f"  t-statistic : {t_stat:.4f}")
print(f"  p-value     : {p_value:.4f}")
if p_value < 0.05:
    direction = "higher" if t_stat > 0 else "lower"
    print(f"  → Statistically significant (p < 0.05): EEGNet is {direction} than CSP+SVM")
else:
    print(f"  → Not statistically significant (p >= 0.05): no clear winner")

### What to look for in this plot:

- **Per-subject comparison**: For each subject, compare the blue (CSP+SVM) and red (EEGNet)
  bars. Subjects where EEGNet wins will have a taller red bar.
- **Mean accuracy lines**: The dotted lines show the overall mean for each method.
  If the red dotted line is above the blue one, EEGNet outperforms CSP+SVM on average.
- **Subject variability**: Both methods should show the same general pattern — subjects
  that are good for CSP+SVM should also be good for EEGNet (strong MI producers).
- **Paired t-test**: A p-value < 0.05 means the accuracy difference is statistically
  significant across subjects. A positive t-statistic means EEGNet is better.
- **Subjects near chance**: Both methods may struggle with "BCI illiterate" subjects —
  deep learning cannot overcome the absence of a neural signal.

In [ ]:
# ============================================================
# SECTION 5 — Cross-dataset: EEGNet on BCI Competition IV 2a
# ============================================================

print("=" * 65)
print("Cross-dataset validation: EEGNet on BCI Competition IV 2a (9 subjects)")
print("=" * 65)

subject_accuracies_eegnet_bci2a = {}  # {subject_id: accuracy}

# Load BCI IV 2a dataset via MOABB
dataset_bci2a = BNCI2014_001()         # BCI Competition IV dataset 2a
paradigm_lr = LeftRightImagery()        # Left vs right hand MI paradigm

for subj_id in range(1, 10):  # 9 subjects
    print(f"\nProcessing BCI IV 2a — Subject {subj_id}...")
    try:
        # MOABB paradigm.get_data returns (n_trials, n_channels, n_timepoints), labels, metadata
        X_bci, y_bci, _ = paradigm_lr.get_data(dataset=dataset_bci2a, subjects=[subj_id])

        # Encode string labels to integers: sorted unique → 0, 1
        unique_labels = sorted(np.unique(y_bci))
        label_map = {lab: idx for idx, lab in enumerate(unique_labels)}
        y_bci_int = np.array([label_map[l] for l in y_bci])

        print(f"  Shape: {X_bci.shape}, Classes: {dict(Counter(y_bci_int))}")

        # Skip if too few trials
        class_counts_bci = Counter(y_bci_int)
        if min(class_counts_bci.values()) < 6:
            print(f"  Skipped — too few trials {dict(class_counts_bci)}")
            continue

        # Train EEGNet from scratch (n_channels=22 for BCI IV 2a)
        acc = train_eegnet_for_subject(
            X_bci, y_bci_int,
            n_channels=X_bci.shape[1],       # 22 channels
            n_timepoints=X_bci.shape[2],     # Time samples at 250 Hz
            n_epochs=300, lr=0.001, patience=20, device=device
        )
        subject_accuracies_eegnet_bci2a[subj_id] = acc
        print(f"  Subject {subj_id}: EEGNet accuracy = {acc:.3f}")

    except Exception as e:
        print(f"  Subject {subj_id}: FAILED — {e}")

print(f"\nProcessed {len(subject_accuracies_eegnet_bci2a)} / 9 subjects")
bci_eegnet_accs = list(subject_accuracies_eegnet_bci2a.values())
print(f"EEGNet mean accuracy (BCI IV 2a): {np.mean(bci_eegnet_accs):.3f} ± {np.std(bci_eegnet_accs):.3f}")

In [ ]:
# ============================================================
# SECTION 5 — Grouped bar chart: CSP+SVM vs EEGNet (BCI IV 2a)
# ============================================================

# Build aligned subject lists for BCI IV 2a
# subject_accuracies_csp_bci2a uses keys like 'BCI2a_S01' or int keys
# subject_accuracies_eegnet_bci2a uses int keys (1..9)

# Map CSP results to integer subject IDs for alignment
csp_bci2a_int = {}  # {int_subject_id: accuracy}
for k, v in subject_accuracies_csp_bci2a.items():
    if isinstance(k, int):
        csp_bci2a_int[k] = v
    else:
        # Extract integer from string key like 'BCI2a_S01'
        sid = int(''.join(filter(str.isdigit, str(k))))
        csp_bci2a_int[sid] = v

# Common subjects with both CSP and EEGNet results
common_bci2a = sorted(set(csp_bci2a_int.keys()) & set(subject_accuracies_eegnet_bci2a.keys()))

csp_bci2a_common = [csp_bci2a_int[s] for s in common_bci2a]
eeg_bci2a_common = [subject_accuracies_eegnet_bci2a[s] for s in common_bci2a]
labels_bci2a = [f'S{s:02d}' for s in common_bci2a]

# --- Grouped bar chart ---
fig, ax = plt.subplots(figsize=(12, 6))

x_pos = np.arange(len(common_bci2a))
bar_width = 0.35

bars_csp = ax.bar(x_pos - bar_width/2, csp_bci2a_common, bar_width,
                  color='#3498db', edgecolor='black', linewidth=0.5, label='CSP+SVM')
bars_eeg = ax.bar(x_pos + bar_width/2, eeg_bci2a_common, bar_width,
                  color='#e74c3c', edgecolor='black', linewidth=0.5, label='EEGNet')

# Add accuracy values on top of bars
for bar, acc in zip(bars_csp, csp_bci2a_common):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold', color='#3498db')
for bar, acc in zip(bars_eeg, eeg_bci2a_common):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold', color='#e74c3c')

# Reference lines
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1.5, label='Chance (50%)')
ax.axhline(np.mean(csp_bci2a_common), color='#3498db', linestyle=':',
           linewidth=1.5, label=f'CSP mean ({np.mean(csp_bci2a_common):.2f})')
ax.axhline(np.mean(eeg_bci2a_common), color='#e74c3c', linestyle=':',
           linewidth=1.5, label=f'EEGNet mean ({np.mean(eeg_bci2a_common):.2f})')

ax.set_xlabel('Subject', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('CSP+SVM vs EEGNet — BCI Competition IV 2a',
             fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(labels_bci2a, fontsize=11)
ax.legend(fontsize=9, loc='upper right')
ax.set_ylim(0.3, 1.0)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/section5_fig5.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/section5_fig5.png")

### What to look for in this plot:

- **Cross-dataset generalization**: EEGNet should maintain competitive performance on
  BCI IV 2a despite having only 22 channels (vs 64 in Schalk) and a different sampling
  rate (250 Hz vs 160 Hz). This confirms the architecture generalizes.
- **BCI IV 2a typically has higher accuracy**: These subjects were well-trained BCI users
  recorded in a controlled setting, so expect 65–85% for both methods.
- **Method comparison**: EEGNet may show a larger advantage on this cleaner dataset because
  it can exploit subtle temporal patterns that CSP's log-variance features miss.
- **Per-subject patterns**: The relative ranking of subjects should be similar for both
  methods — good MI performers are good regardless of the classifier.

In [ ]:
# ============================================================
# SECTION 5 — Final summary
# ============================================================

# Compute summary statistics across all Schalk subjects
eegnet_schalk_accs = list(subject_accuracies_eegnet_schalk.values())
eegnet_mean = np.mean(eegnet_schalk_accs) * 100   # Convert to percentage
eegnet_std = np.std(eegnet_schalk_accs) * 100     # Convert to percentage

# Print comprehensive summary table
print("\n" + "=" * 70)
print("SECTION 5 SUMMARY — EEGNet vs CSP+SVM")
print("=" * 70)

# Schalk dataset summary
csp_schalk_accs = list(subject_accuracies_csp.values())
print(f"\nSchalk Dataset (S001–S109):")
print(f"  CSP+SVM : {np.mean(csp_schalk_accs)*100:.1f}% ± {np.std(csp_schalk_accs)*100:.1f}% "
      f"({len(csp_schalk_accs)} subjects)")
print(f"  EEGNet  : {eegnet_mean:.1f}% ± {eegnet_std:.1f}% "
      f"({len(eegnet_schalk_accs)} subjects)")

# BCI IV 2a summary
if subject_accuracies_eegnet_bci2a:
    bci_eeg_accs = list(subject_accuracies_eegnet_bci2a.values())
    bci_csp_accs = list(subject_accuracies_csp_bci2a.values())
    print(f"\nBCI Competition IV 2a:")
    print(f"  CSP+SVM : {np.mean(bci_csp_accs)*100:.1f}% ± {np.std(bci_csp_accs)*100:.1f}% "
          f"({len(bci_csp_accs)} subjects)")
    print(f"  EEGNet  : {np.mean(bci_eeg_accs)*100:.1f}% ± {np.std(bci_eeg_accs)*100:.1f}% "
          f"({len(bci_eeg_accs)} subjects)")

print("=" * 70)
print(f"Section 5 complete — EEGNet mean accuracy: {eegnet_mean:.1f}% +/- {eegnet_std:.1f}%")